In [3]:
!pip install gensim

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   --- ------------------------------------ 1.8/24.4 MB 10.1 MB/s eta 0:00:03
   ----- ---------------------------------- 3.1/24.4 MB 9.2 MB/s eta 0:00:03
   ------ --------------------------------- 3.9/24.4 MB 6.0 MB/s eta 0:00:04
   ------ --------------------------------- 4.2/24.4 MB 5.2 MB/s eta 0:00:04
   -------- ------------------------------- 5.0/24.4 MB 4.8 MB/s eta 0:00:05
   --------- ------------------------------ 6.0/24.4 MB 4.7 MB/s eta 0:00:04
   ----------- ---------------------------- 6.8/24.4 MB 4.6 MB/s eta 0:00:04
   ------------ --------------------------- 7.6/24.4 MB 4.3 MB/s eta 0:00:04
   ------------- -------------------------- 8.4/24.4 MB 4.3 MB/s eta 0:00:04
   -------------- ------------------------- 8.9/24.4 MB 4.2 MB/s eta 0:00:04
   --------------- ------------------------ 9.7/24.4 MB 4.1 MB/s eta 0:00:04
   ---------------- ----------------------- 10.2/24.4 MB 4.0 MB/s eta 0:00:04
   -


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
!pip install scikit-learn

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.2 MB 322.8 kB/s eta 0:00:24
   -- ------------------------------------- 0.5/8.2 MB 322.8 kB/s eta 0:00:24
   -- ------------------------------------- 0.5/8.2 MB 322.8 kB/s eta 0:00:24
   -- ------------------------------------- 0.5/8.2 MB 322.8 kB/s eta 0:00:24
   -- ------------------------------------- 0.5/8.2 MB 322.8 kB/s eta 0:00:24
   -- ------------------------------------- 0.5/8.2 MB 322.8 kB/s eta 0:00:24
   -- ------------------------------------- 0.5/8.2 MB 322.8 kB/s eta 0:00:24
   -- ----------------------------------


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import json
import pandas as pd
from gensim.models import Word2Vec
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("boxoffice_month_genre.csv", encoding="utf-8-sig")
df = df.dropna(subset=["genre"])

movies = df[["movie_name", "genre"]].drop_duplicates(subset="movie_name").reset_index(drop=True)
print(f"고유 영화 수: {len(movies)}")

# konlpy 없이: 제목을 띄어쓰기로 나누고 + 장르 추가
def make_tokens(row):
    title_tokens = str(row["movie_name"]).replace(":", " ").split()
    genre_tokens = str(row["genre"]).split(",")
    return title_tokens + genre_tokens

movies["tokens"] = movies.apply(make_tokens, axis=1)

# Word2Vec 학습
sentences = movies["tokens"].tolist()
model = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=1, workers=4, sg=1)
print(f"학습된 단어 수: {len(model.wv.key_to_index)}")

def get_movie_vector(tokens):
    vectors = [model.wv[t] for t in tokens if t in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

movies["vector"] = movies["tokens"].apply(get_movie_vector)
movie_vectors = np.stack(movies["vector"].values)
cosine_sim = cosine_similarity(movie_vectors)

def recommend(movie_name, top_n=5):
    idx_list = movies[movies["movie_name"] == movie_name].index
    if len(idx_list) == 0:
        return f"'{movie_name}'을 찾을 수 없습니다."
    idx = idx_list[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [s for s in sim_scores if s[0] != idx][:top_n]
    for i, score in sim_scores:
        print(movies.iloc[i]["movie_name"], "|", movies.iloc[i]["genre"], "| 유사도:", round(score, 3))

print("\n=== '범죄도시3'와 유사한 영화 ===")
recommend("범죄도시3")

print("\n=== '서울의 봄'와 유사한 영화 ===")
recommend("서울의 봄")

고유 영화 수: 10988
학습된 단어 수: 15079

=== '범죄도시3'와 유사한 영화 ===
피버 | 기타 | 유사도: 1.0
재춘언니 | 기타 | 유사도: 1.0
타종 | 기타 | 유사도: 1.0
태양없이 | 기타 | 유사도: 1.0
백탑지광 | 기타 | 유사도: 1.0

=== '서울의 봄'와 유사한 영화 ===
퀴어 | 드라마 | 유사도: 0.999
오월의 눈 | 드라마 | 유사도: 0.999
스틸 플라워 | 드라마 | 유사도: 0.999
검은 난초 | 드라마 | 유사도: 0.999
사람의 아들 | 드라마 | 유사도: 0.999


In [ ]:
!pip install requests

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import os

# !pip install requests beautifulsoup4 (설치 안 되어 있으면 먼저 실행)
os.chdir("H:\&문체부 공모전&")
# 100만 이상 흥행작 목록 (본인이 추려둔 파일명으로 수정)
df = pd.read_csv("top_movies_100만이상.csv", encoding="utf-8-sig")
movie_names = df["movie_name"].tolist()

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

def get_namuwiki_genres(movie_name):
    """나무위키 정보 표에서 '장르' 항목 텍스트만 추출"""
    try:
        url = f"https://namu.wiki/w/{movie_name}"
        res = requests.get(url, headers=headers, timeout=10)
        if res.status_code != 200:
            print(f"  -> 접속 실패 ({res.status_code}): {movie_name}")
            return None
        soup = BeautifulSoup(res.text, "html.parser")
        text = soup.get_text()
        
        # "장르" 단어 뒤에 나오는 텍스트 추출 (보통 정보 표에 "장르 액션, 범죄" 형식)
        match = re.search(r"장르\s*([가-힣A-Za-z,/\s]{2,40})", text)
        if match:
            genre_raw = match.group(1).strip()
            # 쉼표, 슬래시 기준으로 나누고 너무 긴 문자열은 자름
            genres = re.split(r"[,/]", genre_raw)
            genres = [g.strip() for g in genres if g.strip()][:5]  # 최대 5개까지만
            return genres
        return None
    except Exception as e:
        print(f"  -> 에러: {movie_name} - {e}")
        return None

results = {}
for i, movie in enumerate(movie_names):
    print(f"[{i+1}/{len(movie_names)}] {movie} 수집 중...")
    genres = get_namuwiki_genres(movie)
    results[movie] = genres
    time.sleep(1)  # 서버 부담 방지

# 결과를 genre_1, genre_2... 컬럼으로 정리
max_genre_count = max(len(g) for g in results.values() if g) if any(results.values()) else 1
rows = []
for movie, genres in results.items():
    row = {"movie_name": movie}
    if genres:
        for idx, g in enumerate(genres):
            row[f"genre_{idx+1}"] = g
    rows.append(row)

result_df = pd.DataFrame(rows)
result_df.to_csv("movies_namuwiki_genre.csv", index=False, encoding="utf-8-sig")
print("\n저장 완료!")
print(result_df)

In [1]:
import pandas as pd
from gensim.models import Word2Vec
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# !pip install gensim scikit-learn (설치 안 되어 있으면 먼저 실행)

import os
os.chdir(r"H:\&문체부 공모전&")  # 본인 폴더 경로로 수정

df = pd.read_csv("movies_namuwiki_genre.csv", encoding="utf-8-sig")
genre_cols = ["genre_1", "genre_2", "genre_3", "genre_4", "genre_5"]

# 장르 컬럼들을 토큰 리스트로 합치기 (NaN 제외, 공백 제거)
df["tokens"] = df[genre_cols].apply(lambda row: [str(g).strip() for g in row if pd.notna(g)], axis=1)

print("토큰 예시:")
print(df[["movie_name", "tokens"]].head(10))

all_tokens = [t for tokens in df["tokens"] for t in tokens]
unique_genres = set(all_tokens)
print(f"\n전체 장르(단어) 종류: {len(unique_genres)}개")

# Word2Vec 학습
sentences = df["tokens"].tolist()
model = Word2Vec(sentences=sentences, vector_size=50, window=5, min_count=1, workers=4, sg=1)
print(f"학습된 단어 수: {len(model.wv.key_to_index)}")

# 영화별 벡터 = 토큰들의 벡터 평균
def get_movie_vector(tokens):
    vectors = [model.wv[t] for t in tokens if t in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

df["vector"] = df["tokens"].apply(get_movie_vector)
movie_vectors = np.stack(df["vector"].values)

# 전체 영화 간 코사인 유사도 행렬
cosine_sim = cosine_similarity(movie_vectors)

# 추천 함수
def recommend(movie_name, top_n=5):
    idx_list = df[df["movie_name"] == movie_name].index
    if len(idx_list) == 0:
        print(f"'{movie_name}'을 찾을 수 없습니다.")
        return
    idx = idx_list[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [s for s in sim_scores if s[0] != idx][:top_n]
    for i, score in sim_scores:
        print(f"  {df.iloc[i]['movie_name']} | {df.iloc[i]['tokens']} | 유사도: {round(score, 3)}")

# 테스트
print("\n=== '범죄도시3'와 유사한 영화 ===")
recommend("범죄도시3")

print("\n=== '서울의 봄'와 유사한 영화 ===")
recommend("서울의 봄")

print("\n=== '안시성'와 유사한 영화 ===")
recommend("안시성")

print("\n=== '정직한 후보'와 유사한 영화 ===")
recommend("정직한 후보")

토큰 예시:
  movie_name                       tokens
0         명량            [사극, 액션, 전쟁, 드라마]
1   왕과 사는 남자                    [사극, 드라마]
2       극한직업            [코미디, 액션, 범죄, 형사]
3       국제시장      [시대극, 드라마, 전쟁, 코미디, 가족]
4     겨울왕국 2  [애니메이션, 어드벤처, 코미디, 가족, 판타지]
5    7번방의 선물      [코미디, 드라마, 법정, 시대극, 휴먼]
6        알라딘  [판타지, 뮤지컬, 로맨틱 코미디, 모험, 액션]
7         암살      [액션, 드라마, 첩보, 시대극, 스릴러]
8      택시운전사       [휴먼, 드라마, 가족, 모험, 시대극]
9         파묘     [공포, 미스터리, 오컬트, 스릴러, 퇴마]

전체 장르(단어) 종류: 95개
학습된 단어 수: 95

=== '범죄도시3'와 유사한 영화 ===
  범죄도시4 | ['액션', '범죄', '스릴러', '느와르', '코미디'] | 유사도: 1.0
  분노의 질주: 라이드 오어 다이 | ['범죄', '액션', '스릴러'] | 유사도: 0.837
  존 윅 4 | ['액션', '느와르', '스릴러', '범죄', '복수'] | 유사도: 0.811
  히트맨2 | ['액션', '코미디', '어드벤처', '스릴러'] | 유사도: 0.703
  나우 유 씨 미 3 | ['범죄', '액션'] | 유사도: 0.693

=== '서울의 봄'와 유사한 영화 ===
  남산의 부장들 | ['시대극', '정치', '드라마', '스릴러', '느와르'] | 유사도: 0.828
  탈주 | ['액션', '어드벤처', '스릴러', '밀리터리', '드라마'] | 유사도: 0.8
  타짜 | ['드라마', '범죄', '스릴러', '느와르', '서스펜스'] | 유사도: 0.758
  타짜: 원 아이드 잭 | ['범죄

In [4]:
import pandas as pd
from gensim.models import Word2Vec
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

import os
os.chdir(r"H:\&문체부 공모전&")  # 본인 폴더 경로로 수정

df = pd.read_csv("movies_namuwiki_genre.csv", encoding="utf-8-sig")
genre_cols = ["genre_1", "genre_2", "genre_3", "genre_4", "genre_5"]

df["tokens"] = df[genre_cols].apply(lambda row: [str(g).strip() for g in row if pd.notna(g)], axis=1)

# Word2Vec 학습
sentences = df["tokens"].tolist()
model = Word2Vec(sentences=sentences, vector_size=50, window=5, min_count=1, workers=4, sg=1)
print(f"학습된 단어 수: {len(model.wv.key_to_index)}")

# genre_1부터 순서대로 가중치 부여 (가중치 합이 1이 되도록 정규화)
def get_weighted_movie_vector(tokens):
    valid_tokens = [t for t in tokens if t in model.wv]
    if len(valid_tokens) == 0:
        return np.zeros(model.vector_size)
    
    n = len(valid_tokens)
    raw_weights = [n - i for i in range(n)]  # 예: 5개면 [5,4,3,2,1]
    total = sum(raw_weights)
    weights = [w / total for w in raw_weights]  # 합이 1이 되도록 정규화
    
    vectors = [model.wv[t] for t in valid_tokens]
    weighted_vector = np.sum([w * v for w, v in zip(weights, vectors)], axis=0)
    return weighted_vector

df["tokens_weights"] = df["tokens"].apply(
    lambda tokens: [round(w, 3) for w in 
        ([(len([t for t in tokens if t in model.wv]) - i) / sum(range(1, len([t for t in tokens if t in model.wv]) + 1)) 
          for i in range(len([t for t in tokens if t in model.wv]))])]
)

df["vector"] = df["tokens"].apply(get_weighted_movie_vector)
movie_vectors = np.stack(df["vector"].values)
cosine_sim = cosine_similarity(movie_vectors)

def recommend(movie_name, top_n=5):
    idx_list = df[df["movie_name"] == movie_name].index
    if len(idx_list) == 0:
        print(f"'{movie_name}'을 찾을 수 없습니다.")
        return
    idx = idx_list[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [s for s in sim_scores if s[0] != idx][:top_n]
    for i, score in sim_scores:
        print(f"  {df.iloc[i]['movie_name']} | {df.iloc[i]['tokens']} | 유사도: {round(score, 3)}")

print("\n=== '범죄도시3'와 유사한 영화 (가중치 적용) ===")
recommend("범죄도시3")

print("\n=== '서울의 봄'와 유사한 영화 (가중치 적용) ===")
recommend("서울의 봄")

print("\n=== '안시성'와 유사한 영화 (가중치 적용) ===")
recommend("안시성")

print("\n=== '정직한 후보'와 유사한 영화 (가중치 적용) ===")
recommend("정직한 후보")


학습된 단어 수: 95

=== '범죄도시3'와 유사한 영화 (가중치 적용) ===
  범죄도시4 | ['액션', '범죄', '스릴러', '느와르', '코미디'] | 유사도: 0.987
  분노의 질주: 라이드 오어 다이 | ['범죄', '액션', '스릴러'] | 유사도: 0.95
  베테랑2 | ['범죄', '액션', '스릴러', '블랙 코미디', '형사'] | 유사도: 0.925
  조작된 도시 | ['범죄', '액션', '스릴러', '미스터리', '복수'] | 유사도: 0.911
  존 윅 4 | ['액션', '느와르', '스릴러', '범죄', '복수'] | 유사도: 0.861

=== '서울의 봄'와 유사한 영화 (가중치 적용) ===
  국가부도의 날 | ['드라마', '시대극'] | 유사도: 0.735
  남산의 부장들 | ['시대극', '정치', '드라마', '스릴러', '느와르'] | 유사도: 0.708
  말모이 | ['드라마', '시대극', '역사', '코미디'] | 유사도: 0.707
  국제시장 | ['시대극', '드라마', '전쟁', '코미디', '가족'] | 유사도: 0.703
  하얼빈 | ['시대극', '드라마', '전기', '액션', '첩보'] | 유사도: 0.686

=== '안시성'와 유사한 영화 (가중치 적용) ===
  한산: 용의 출현 | ['사극', '액션', '드라마', '스릴러'] | 유사도: 0.939
  노량: 죽음의 바다 | ['사극', '액션', '드라마', '스릴러', '전쟁'] | 유사도: 0.939
  명량 | ['사극', '액션', '전쟁', '드라마'] | 유사도: 0.935
  군도: 민란의 시대 | ['사극', '액션', '드라마'] | 유사도: 0.925
  콘스탄틴 | ['액션', '스릴러', '드라마', '판타지'] | 유사도: 0.734

=== '정직한 후보'와 유사한 영화 (가중치 적용) ===
  파일럿 | ['코미디', '드라마', '휴먼', '사회고발'] | 유사도: 0.655
 